# 08 — Gemma 4 E4B (LLM locale via ollama)

Secondo LLM locale del benchmark (vedi il notebook [07_qwen.ipynb](07_qwen.ipynb) per
l'impostazione generale). **Gemma 4 E4B** è il modello "laptop-tier" di Google.

**Perché `MAX_TOKENS=1500` (gli altri modelli usano 200/300)**: Gemma 4 è un modello con
*reasoning* e i token di ragionamento consumano il budget di `max_tokens` **prima** che inizi
la risposta visibile → con il limite 300 della corsa originale via LM Studio la richiesta
terminava con `finish_reason=length` e `content` vuoto (il prefisso `/no_think` non ha effetto
su gemma). Nella corsa originale questo produceva una risposta **vuota in 81 casi su 100**, e
i file allora importati coprivano solo 19 esempi. Con il budget alzato a **1500** la corsa
ollama di questo notebook copre **tutti i 100 esempi** — il riassunto visibile resta comunque
di lunghezza paragonabile agli altri modelli (~200–300 token) — quindi gemma partecipa ora al
confronto del notebook [05_confronto.ipynb](05_confronto.ipynb) a copertura piena, senza più
il caveat storico di copertura parziale.

## Provenienza dei risultati committati e ripresa

I file committati (`results/summaries/gemma_sample.tsv` e le metriche) provengono dalla
**corsa di questo notebook via ollama** (100/100 esempi, 2026-07-16), che ha rigenerato da
zero i riassunti sostituendo i 19 esempi importati dalla corsa originale di Federica via
LM Studio (Mac M4, 2026-07-16; i CSV originali restano archiviati in `notebooks/llm/`).
Rispetto alla corsa originale ci sono tre differenze deliberate:

- il documento passa da `su.prepara_documento` (separatore `` ||||| `` → newline), mentre la
  corsa originale inviava il testo grezzo con il separatore incluso;
- niente `extra_body={"enable_thinking": False}` (parametro specifico di LM Studio; ollama
  lo ignorerebbe);
- `MAX_TOKENS=1500` invece di 300, per lasciare spazio al reasoning di Gemma 4 (vedi sopra:
  con 300 la risposta visibile resta vuota).

⚠️ **Attenzione alla ripresa**: il ciclo condiviso salta i `row_id` già presenti nel TSV, quindi
rieseguire la generazione su un TSV esistente **aggiunge solo le righe mancanti** — utile per
completare una corsa ollama interrotta, ma da evitare su un file prodotto da un backend o con
una configurazione diversi (es. un re-import della corsa LM Studio), perché mescolerebbe le due
corse nello stesso file. In quel caso **eliminare prima** `results/summaries/gemma_sample.tsv`
e rigenerare tutto (rieseguendo poi la valutazione).

In [1]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer
try:
    import openai  # noqa: F401
except ImportError:
    %pip install openai

In [2]:
# --- Configurazione ---------------------------------------------------------
import summ_utils as su

METODO     = 'gemma'
SCOPE      = 'sample'    # questo notebook gira solo sul campione
N_SAMPLES  = 100         # deve combaciare con il campione creato dal notebook 00
SEED       = 42
LIMIT      = None        # es. 3 per uno smoke test rapido; None = tutti
MODELLO    = 'gemma4:latest'  # tag ollama; verificare/adattare con `ollama list`
OLLAMA_URL = 'http://localhost:11434/v1'   # endpoint OpenAI-compatibile di ollama

# La corsa LM Studio originale usava 300, ma il reasoning di Gemma 4 consuma il
# budget prima della risposta visibile (81 risposte vuote su 100, verificato
# anche via ollama: finish_reason=length): 1500 lascia spazio a entrambi.
MAX_TOKENS  = 1500
TEMPERATURE = 0.3
PROMPT_SYSTEM = ('You are a helpful assistant that summarizes news articles '
                 'from different sources concisely.')
PROMPT_USER   = ('/no_think\n\n'
                 'Summarize the following document into a comprehensive summary: {documento}')
ETICHETTA   = 'Gemma '
NOTE_CONFIG = ('prompt zero-shot in inglese; max_tokens=1500 invece dei 300 originali '
               'perche\' il reasoning di Gemma 4 consumava il budget (81 risposte '
               'vuote su 100 nella corsa LM Studio)')

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'

print(f'Modello : {MODELLO} via {OLLAMA_URL}')
print(f'Output  : {OUT_PATH}')

Modello : gemma4:latest via http://localhost:11434/v1
Output  : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\summaries\gemma_sample.tsv


## Generazione dei riassunti

Il modello gira **fuori dal notebook**, nel server locale di ollama: qui usiamo solo il client
`openai` puntato all'endpoint OpenAI-compatibile (`http://localhost:11434/v1`). Prerequisiti:

```bash
ollama pull gemma4
ollama serve   # se il servizio non e' gia' attivo
```

Il ciclo e la scrittura incrementale (con ripresa) sono quelli condivisi di `summ_utils`; una
risposta vuota del modello solleva un'eccezione, così la riga viene registrata come errore e
**non** scritta nel TSV (ritentabile alla corsa successiva).

In [3]:
from openai import OpenAI

client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')  # la chiave e' ignorata da ollama

def genera(documento):
    risposta = client.chat.completions.create(
        model=MODELLO,
        messages=[{'role': 'system', 'content': PROMPT_SYSTEM},
                  {'role': 'user', 'content': PROMPT_USER.format(documento=documento)}],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE)
    scelta = risposta.choices[0]
    contenuto = scelta.message.content
    if not contenuto or not contenuto.strip():
        # solleva -> il ciclo condiviso registra l'errore e NON scrive la riga
        raise RuntimeError(f'risposta vuota (finish_reason={scelta.finish_reason})')
    return contenuto.strip()

esempi    = su.carica_campione(SAMPLE_PATH)
scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT,
                                etichetta=ETICHETTA)
scrittore.chiudi()

Gemma [1] media 22.3 s/esempio (saltati 0 gia' fatti)
Gemma [2] media 18.9 s/esempio (saltati 0 gia' fatti)
Gemma [3] media 17.6 s/esempio (saltati 0 gia' fatti)
Gemma [10] media 14.9 s/esempio (saltati 0 gia' fatti)
Gemma [20] media 15.1 s/esempio (saltati 0 gia' fatti)
Gemma [30] media 15.0 s/esempio (saltati 0 gia' fatti)
Gemma [40] media 14.6 s/esempio (saltati 0 gia' fatti)
Gemma [50] media 14.5 s/esempio (saltati 0 gia' fatti)
Gemma [60] media 14.3 s/esempio (saltati 0 gia' fatti)
Gemma [70] media 14.2 s/esempio (saltati 0 gia' fatti)
Gemma [80] media 14.2 s/esempio (saltati 0 gia' fatti)
Gemma [90] media 14.4 s/esempio (saltati 0 gia' fatti)
Gemma [100] media 14.5 s/esempio (saltati 0 gia' fatti)
Gemma Completato: 100 nuovi, 0 saltati, 0 errori, 1450 s totali


## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del
benchmark. Output: `results/metrics/gemma_sample_per_example.csv` e `…_aggregate.json`.

⚠️ La valutazione misura il TSV salvato **qualunque ne sia l'origine**, mentre il JSON aggregato
riporta la configurazione dichiarata qui sotto: oggi le due cose coincidono (il TSV committato
è la corsa ollama di questo notebook), ma se il TSV venisse re-importato da un'altra corsa
(es. LM Studio via `scripts/import_llm_results.py`, che scrive la propria provenienza nel JSON),
rieseguire questa cella sovrascriverebbe quella provenienza. Rieseguirla solo su un TSV
generato con la configurazione qui sopra.

In [4]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = su.carica_campione(SAMPLE_PATH)
config = {'modello': MODELLO,
          'backend': 'ollama (endpoint OpenAI-compatibile)',
          'max_tokens': MAX_TOKENS, 'temperature': TEMPERATURE,
          'prompt_system': PROMPT_SYSTEM, 'prompt_user': PROMPT_USER,
          'note': NOTE_CONFIG}

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))
print('\nMedie per split:')
for split, valori in aggregato['per_split'].items():
    print(f"  {split:5s} (n={valori['n_esempi']}): ROUGE-1 F1 = {valori['rouge1_f1']:.3f}")

c:\Users\antonio.girasella\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Metriche per-esempio : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\gemma_sample_per_example.csv (100 righe)
Metriche aggregate   : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\gemma_sample_aggregate.json
{
  "rouge1_f1": 0.33087541383259145,
  "rouge1_p": 0.30416961499585754,
  "rouge1_r": 0.38031591056910324,
  "rouge2_f1": 0.09399870219918664,
  "rouge2_p": 0.08749620485363634,
  "rouge2_r": 0.10852657618330216,
  "rougeL_f1": 0.1743615944122483,
  "rougeL_p": 0.1624246945435129,
  "rougeL_r": 0.2016388072074782,
  "bleu": 0.060800711108134,
  "meteor": 0.42730224108796117,
  "parole_generate": 291.9
}

Medie per split:
  test  (n=8): ROUGE-1 F1 = 0.341
  train (n=81): ROUGE-1 F1 = 0.328
  val   (n=11): ROUGE-1 F1 = 0.346


## Ispezione qualitativa

In [5]:
su.mostra_esempi(su.carica_campione(SAMPLE_PATH), riassunti, quanti=2)

--- row_id=425 (split=train) ---
GENERATO   : A campaign mailer sent by Republican state Senate candidate Ed Charamut in Connecticut has drawn widespread condemnation for allegedly using anti-Semitic tropes to attack his Democratic opponent, State Representative Matthew Lesser, who is Jewish.

**The Incident:**
The double-sided mailer depicts Rep. Lesser with an altered image—specifically featuring large eyes and a caricature of him holding a wad of cash—alongside the tagline: "Matt Lesser will take everything you worked for." The campaign'
RIFERIMENTO: Just days after the slaying of 11 Jewish congregants at a Pittsburgh synagogue, a GOP candidate for a state Senate seat in Connecticut is accused of sending a mailer using an "age-old anti-Semitic trope." The ad sent out by Ed Charamut includes what the Washington Post calls a "money-grubbing" picture (here) of smiling opponent Matt Lesser, clutching $100 bills with a "crazed look in his eyes." Lesser says the original image of him was 